# Train & Build the Goal-Oriented RL Chatbot from Scratch

This notebook gives you **full control** over every piece of the training pipeline:

| Section | What you can do |
|---|---|
| **Configuration** | Tweak all hyperparameters (learning rate, gamma, epsilon, hidden size, …) |
| **Model Architecture** | Edit the DQN neural network (add layers, change activations, switch to DDQN, …) |
| **Reward Function** | Modify the reward signal |
| **Warmup** | Fill the replay memory with rule-based episodes |
| **Training Loop** | Train with live metrics & plots |
| **Evaluation** | Test the trained agent and visualise results |
| **Save / Load** | Persist and restore model weights |

---
## 1. Imports

In [ ]:
import os, sys, json, pickle, copy, random, re, math, time
import numpy as np

# Plotting
try:
    import matplotlib
    matplotlib.use('inline')  # safe for notebooks
except Exception:
    pass
import matplotlib.pyplot as plt
%matplotlib inline

# Ensure repo root is on path
REPO_ROOT = os.path.dirname(os.path.abspath('__file__'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from keras.models import Sequential
from keras.layers import Dense
from keras.optimizers import Adam

from user_simulator import UserSimulator
from error_model_controller import ErrorModelController
from state_tracker import StateTracker
from db_query import DBQuery
from utils import remove_empty_slots, convert_list_to_dict, reward_function
from dialogue_config import (
    all_intents, all_slots, agent_actions, agent_inform_slots, agent_request_slots,
    rule_requests, usersim_default_key, FAIL, SUCCESS, NO_OUTCOME,
    no_query_keys
)

print('All imports successful.')

---
## 2. Configuration — Edit Freely

All tuneable hyper-parameters are gathered here in **one place**. Change any value and re-run the cells below.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║                   HYPERPARAMETERS                           ║
# ╚══════════════════════════════════════════════════════════════╝

config = {
    # --- Data paths ---
    'db_file_paths': {
        'database':   'data/movie_db.pkl',
        'dict':       'data/movie_dict.pkl',
        'user_goals': 'data/movie_user_goals.pkl',
    },

    # --- Run / training schedule ---
    'run': {
        'usersim':                True,
        'warmup_mem':             1000,      # Number of warmup steps to fill memory
        'num_ep_run':             4000,      # Total training episodes  (reduce for quick experiments)
        'train_freq':             100,       # Train & evaluate every N episodes
        'max_round_num':          20,        # Max dialogue turns per episode
        'success_rate_threshold': 0.30,      # Min success rate to flush memory
    },

    # --- DQN Agent ---
    'agent': {
        'load_weights_file_path': '',        # Set to e.g. 'weights/model.h5' to resume training
        'save_weights_file_path': 'weights/model.h5',  # Where to save best weights
        'vanilla':          True,            # True = DQN, False = Double DQN (DDQN)
        'learning_rate':    1e-3,
        'batch_size':       16,
        'dqn_hidden_size':  80,              # ← Try 128, 256, add layers in Sec. 4
        'epsilon_init':     0.1,             # Initial exploration rate
        'epsilon_min':      0.0,             # Minimum epsilon after decay
        'epsilon_decay':    0.0001,          # Per-episode decay (new feature)
        'gamma':            0.9,             # Discount factor
        'max_mem_size':     50000,           # Replay buffer capacity
    },

    # --- Error Model Controller ---
    'emc': {
        'slot_error_mode': 0,
        'slot_error_prob': 0.05,
        'intent_error_prob': 0.0,
    },
}

# Pretty-print for verification
print(json.dumps(config, indent=2))

---
## 3. Load Data

In [ ]:
database   = pickle.load(open(config['db_file_paths']['database'],   'rb'), encoding='latin1')
db_dict    = pickle.load(open(config['db_file_paths']['dict'],       'rb'), encoding='latin1')
user_goals = pickle.load(open(config['db_file_paths']['user_goals'], 'rb'), encoding='latin1')

remove_empty_slots(database)

print(f'Database entries : {len(database)}')
print(f'User goals       : {len(user_goals)}')
print(f'Dict slot keys   : {list(db_dict.keys())}')
print(f'Sample DB entry  : {list(database.values())[0]}')

---
## 4. Model Architecture — Customise the DQN

The class below is a **self-contained, editable** version of `DQNAgent`.  
Change the network, the training loop, the exploration strategy — anything you like.  

### Ideas to try
- Add more hidden layers / change widths
- Use `Dropout`, `BatchNormalization`
- Switch activation to `tanh`, `elu`, `swish`
- Implement prioritised experience replay
- Implement epsilon annealing (skeleton included)

In [ ]:
class CustomDQNAgent:
    """
    Editable DQN Agent for the goal-oriented chatbot.
    Modify _build_model() to change the architecture.
    """

    def __init__(self, state_size, constants):
        self.C = constants['agent']
        self.memory = []
        self.memory_index = 0
        self.max_memory_size = self.C['max_mem_size']
        self.eps = self.C['epsilon_init']
        self.eps_min = self.C.get('epsilon_min', 0.0)
        self.eps_decay = self.C.get('epsilon_decay', 0.0)
        self.vanilla = self.C['vanilla']
        self.lr = self.C['learning_rate']
        self.gamma = self.C['gamma']
        self.batch_size = self.C['batch_size']
        self.hidden_size = self.C['dqn_hidden_size']

        self.load_weights_file_path = self.C['load_weights_file_path']
        self.save_weights_file_path = self.C['save_weights_file_path']

        if self.max_memory_size < self.batch_size:
            raise ValueError('Max memory size must be at least as great as batch size!')

        self.state_size = state_size
        self.possible_actions = agent_actions
        self.num_actions = len(self.possible_actions)
        self.rule_request_set = rule_requests

        # ── Build two networks: behaviour + target ──
        self.beh_model = self._build_model()
        self.tar_model = self._build_model()

        self._load_weights()
        self.reset()

    # ╔══════════════════════════════════════════════════════════╗
    # ║  EDIT THIS METHOD TO CHANGE THE NEURAL NETWORK          ║
    # ╚══════════════════════════════════════════════════════════╝
    def _build_model(self):
        """
        Build and compile the DQN model.
        
        ★ CUSTOMISE HERE ★
        - Add/remove Dense layers
        - Change activation functions ('relu', 'tanh', 'elu', 'swish')
        - Add Dropout, BatchNormalization, etc.
        - Change the optimizer or loss
        """
        model = Sequential()
        # --- Layer 1: Input → Hidden ---
        model.add(Dense(self.hidden_size, input_dim=self.state_size, activation='relu'))
        # --- (Optional) Layer 2: Add more capacity ---
        # model.add(Dense(self.hidden_size, activation='relu'))
        # model.add(Dropout(0.2))
        # --- Output layer ---
        model.add(Dense(self.num_actions, activation='linear'))
        model.compile(loss='mse', optimizer=Adam(lr=self.lr))
        return model

    # ╔══════════════════════════════════════════════════════════╗
    # ║  EDIT THIS TO CHANGE THE REWARD FUNCTION                ║
    # ╚══════════════════════════════════════════════════════════╝
    @staticmethod
    def custom_reward(success, max_round):
        """
        Custom reward function. Default matches the original.
        
        ★ CUSTOMISE HERE ★
        - Increase/decrease penalties & bonuses
        - Add shaping rewards (e.g., small bonus for each inform matched)
        """
        reward = -1  # step penalty
        if success == FAIL:
            reward += -max_round
        elif success == SUCCESS:
            reward += 2 * max_round
        return reward

    # ── Epsilon decay (called after each episode) ──
    def decay_epsilon(self):
        if self.eps > self.eps_min:
            self.eps = max(self.eps_min, self.eps - self.eps_decay)

    # ── Core agent methods (same logic as original) ──
    def reset(self):
        self.rule_current_slot_index = 0
        self.rule_phase = 'not done'

    def get_action(self, state, use_rule=False):
        if self.eps > random.random():
            index = random.randint(0, self.num_actions - 1)
            action = self._map_index_to_action(index)
            return index, action
        else:
            if use_rule:
                return self._rule_action()
            else:
                return self._dqn_action(state)

    def _rule_action(self):
        if self.rule_current_slot_index < len(self.rule_request_set):
            slot = self.rule_request_set[self.rule_current_slot_index]
            self.rule_current_slot_index += 1
            rule_response = {'intent': 'request', 'inform_slots': {}, 'request_slots': {slot: 'UNK'}}
        elif self.rule_phase == 'not done':
            rule_response = {'intent': 'match_found', 'inform_slots': {}, 'request_slots': {}}
            self.rule_phase = 'done'
        elif self.rule_phase == 'done':
            rule_response = {'intent': 'done', 'inform_slots': {}, 'request_slots': {}}
        else:
            raise Exception('Rule action error')
        index = self._map_action_to_index(rule_response)
        return index, rule_response

    def _dqn_action(self, state):
        index = np.argmax(self._dqn_predict_one(state))
        action = self._map_index_to_action(index)
        return index, action

    def _map_action_to_index(self, response):
        for i, action in enumerate(self.possible_actions):
            if response == action:
                return i
        raise ValueError(f'Action not found: {response}')

    def _map_index_to_action(self, index):
        return copy.deepcopy(self.possible_actions[index])

    def _dqn_predict_one(self, state, target=False):
        return self._dqn_predict(state.reshape(1, self.state_size), target=target).flatten()

    def _dqn_predict(self, states, target=False):
        if target:
            return self.tar_model.predict(states)
        else:
            return self.beh_model.predict(states)

    def add_experience(self, state, action, reward, next_state, done):
        if len(self.memory) < self.max_memory_size:
            self.memory.append(None)
        self.memory[self.memory_index] = (state, action, reward, next_state, done)
        self.memory_index = (self.memory_index + 1) % self.max_memory_size

    def empty_memory(self):
        self.memory = []
        self.memory_index = 0

    def is_memory_full(self):
        return len(self.memory) == self.max_memory_size

    def train(self):
        num_batches = len(self.memory) // self.batch_size
        for b in range(num_batches):
            batch = random.sample(self.memory, self.batch_size)
            states      = np.array([s[0] for s in batch])
            next_states = np.array([s[3] for s in batch])

            beh_state_preds = self._dqn_predict(states)
            if not self.vanilla:
                beh_next_states_preds = self._dqn_predict(next_states)
            tar_next_state_preds = self._dqn_predict(next_states, target=True)

            inputs  = np.zeros((self.batch_size, self.state_size))
            targets = np.zeros((self.batch_size, self.num_actions))

            for i, (s, a, r, s_, d) in enumerate(batch):
                t = beh_state_preds[i]
                if not self.vanilla:
                    t[a] = r + self.gamma * tar_next_state_preds[i][np.argmax(beh_next_states_preds[i])] * (not d)
                else:
                    t[a] = r + self.gamma * np.amax(tar_next_state_preds[i]) * (not d)
                inputs[i]  = s
                targets[i] = t

            self.beh_model.fit(inputs, targets, epochs=1, verbose=0)

    def copy(self):
        self.tar_model.set_weights(self.beh_model.get_weights())

    def save_weights(self):
        if not self.save_weights_file_path:
            return
        beh_path = re.sub(r'\.h5', r'_beh.h5', self.save_weights_file_path)
        tar_path = re.sub(r'\.h5', r'_tar.h5', self.save_weights_file_path)
        self.beh_model.save_weights(beh_path)
        self.tar_model.save_weights(tar_path)
        print(f'  Weights saved → {beh_path}, {tar_path}')

    def _load_weights(self):
        if not self.load_weights_file_path:
            return
        beh_path = re.sub(r'\.h5', r'_beh.h5', self.load_weights_file_path)
        tar_path = re.sub(r'\.h5', r'_tar.h5', self.load_weights_file_path)
        self.beh_model.load_weights(beh_path)
        self.tar_model.load_weights(tar_path)
        print(f'  Weights loaded ← {beh_path}, {tar_path}')


print('CustomDQNAgent class defined.')

---
## 5. Initialise Components

In [ ]:
user_sim      = UserSimulator(user_goals, config, database)
emc           = ErrorModelController(db_dict, config)
state_tracker = StateTracker(database, config)

STATE_SIZE     = state_tracker.get_state_size()
MAX_ROUND_NUM  = config['run']['max_round_num']
WARMUP_MEM     = config['run']['warmup_mem']
NUM_EP_TRAIN   = config['run']['num_ep_run']
TRAIN_FREQ     = config['run']['train_freq']
SUCCESS_RATE_THRESHOLD = config['run']['success_rate_threshold']

dqn_agent = CustomDQNAgent(STATE_SIZE, config)

print(f'State size       : {STATE_SIZE}')
print(f'Num actions      : {dqn_agent.num_actions}')
print(f'Warmup steps     : {WARMUP_MEM}')
print(f'Training episodes: {NUM_EP_TRAIN}')
print(f'Train frequency  : {TRAIN_FREQ}')

In [ ]:
# Peek at the model architecture
dqn_agent.beh_model.summary()

---
## 6. Helper Functions

In [ ]:
def episode_reset():
    """Reset state tracker, user sim, and agent for a new episode."""
    state_tracker.reset()
    user_action = user_sim.reset()
    emc.infuse_error(user_action)
    state_tracker.update_state_user(user_action)
    dqn_agent.reset()


def run_round(state, warmup=False):
    """Execute one turn of dialogue. Returns (next_state, reward, done, success)."""
    agent_action_index, agent_action = dqn_agent.get_action(state, use_rule=warmup)
    state_tracker.update_state_agent(agent_action)
    user_action, reward, done, success = user_sim.step(agent_action)
    if not done:
        emc.infuse_error(user_action)
    state_tracker.update_state_user(user_action)
    next_state = state_tracker.get_state(done)
    dqn_agent.add_experience(state, agent_action_index, reward, next_state, done)
    return next_state, reward, done, success

print('Helper functions defined.')

---
## 7. Warmup Phase

Fill the replay memory using the rule-based policy so the DQN has data to learn from on the first training step.

In [ ]:
print('Warmup started...')
total_step = 0
warmup_start = time.time()

while total_step < WARMUP_MEM and not dqn_agent.is_memory_full():
    episode_reset()
    done = False
    state = state_tracker.get_state()
    while not done:
        next_state, _, done, _ = run_round(state, warmup=True)
        total_step += 1
        state = next_state

warmup_elapsed = time.time() - warmup_start
print(f'Warmup finished.  Steps: {total_step}  Memory size: {len(dqn_agent.memory)}  Time: {warmup_elapsed:.1f}s')

---
## 8. Training Loop (with live metrics)

Run the cell below to train the agent. Metrics (success rate, avg reward, epsilon) are recorded every `TRAIN_FREQ` episodes and plotted live.

> **Tip:** Reduce `NUM_EP_TRAIN` in Section 2 for a quick sanity-check run.

In [ ]:
# ── Training metrics storage ──
history_success_rate = []
history_avg_reward   = []
history_epsilon      = []
history_episode      = []

print(f'Training for {NUM_EP_TRAIN} episodes (eval every {TRAIN_FREQ}) ...')
train_start = time.time()

episode = 0
period_reward_total  = 0
period_success_total = 0
success_rate_best    = 0.0

while episode < NUM_EP_TRAIN:
    episode_reset()
    episode += 1
    done = False
    state = state_tracker.get_state()

    while not done:
        next_state, reward, done, success = run_round(state)
        period_reward_total += reward
        state = next_state

    period_success_total += success
    dqn_agent.decay_epsilon()  # ← epsilon annealing

    # ── Periodic evaluation & training ──
    if episode % TRAIN_FREQ == 0:
        success_rate = period_success_total / TRAIN_FREQ
        avg_reward   = period_reward_total / TRAIN_FREQ

        history_episode.append(episode)
        history_success_rate.append(success_rate)
        history_avg_reward.append(avg_reward)
        history_epsilon.append(dqn_agent.eps)

        improved = ''
        if success_rate >= success_rate_best and success_rate >= SUCCESS_RATE_THRESHOLD:
            dqn_agent.empty_memory()
        if success_rate > success_rate_best:
            success_rate_best = success_rate
            dqn_agent.save_weights()
            improved = ' ★ NEW BEST'

        print(f'  Ep {episode:>6} | SR {success_rate:.2%} | Avg R {avg_reward:+.2f} | ε {dqn_agent.eps:.4f}{improved}')

        period_success_total = 0
        period_reward_total  = 0

        dqn_agent.copy()   # target ← behaviour
        dqn_agent.train()  # train behaviour network

train_elapsed = time.time() - train_start
print(f'\nTraining complete. Best success rate: {success_rate_best:.2%}  Time: {train_elapsed:.1f}s')

---
## 9. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Success rate
axes[0].plot(history_episode, history_success_rate, 'b-o', markersize=3)
axes[0].set_title('Success Rate')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Success Rate')
axes[0].set_ylim(-0.05, 1.05)
axes[0].grid(True, alpha=0.3)

# Average reward
axes[1].plot(history_episode, history_avg_reward, 'r-o', markersize=3)
axes[1].set_title('Average Reward')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Avg Reward')
axes[1].grid(True, alpha=0.3)

# Epsilon
axes[2].plot(history_episode, history_epsilon, 'g-o', markersize=3)
axes[2].set_title('Epsilon (Exploration Rate)')
axes[2].set_xlabel('Episode')
axes[2].set_ylabel('Epsilon')
axes[2].set_ylim(-0.05, max(history_epsilon) + 0.05 if history_epsilon else 1.05)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 10. Evaluate the Trained Agent

In [ ]:
# Set epsilon to 0 for pure exploitation during evaluation
original_eps = dqn_agent.eps
dqn_agent.eps = 0.0

NUM_EVAL = 500
eval_successes = 0
eval_rewards = []
eval_lengths = []

for _ in range(NUM_EVAL):
    episode_reset()
    done = False
    ep_reward = 0
    ep_len = 0
    state = state_tracker.get_state()
    while not done:
        agent_action_index, agent_action = dqn_agent.get_action(state)
        state_tracker.update_state_agent(agent_action)
        user_action, reward, done, success = user_sim.step(agent_action)
        ep_reward += reward
        ep_len += 1
        if not done:
            emc.infuse_error(user_action)
        state_tracker.update_state_user(user_action)
        state = state_tracker.get_state(done)
    eval_successes += int(success)
    eval_rewards.append(ep_reward)
    eval_lengths.append(ep_len)

dqn_agent.eps = original_eps

print(f'Evaluation over {NUM_EVAL} episodes:')
print(f'  Success rate : {eval_successes / NUM_EVAL:.2%}')
print(f'  Avg reward   : {np.mean(eval_rewards):.2f} ± {np.std(eval_rewards):.2f}')
print(f'  Avg turns    : {np.mean(eval_lengths):.1f}')

In [ ]:
# Distribution of episode rewards
plt.figure(figsize=(10, 4))
plt.hist(eval_rewards, bins=30, edgecolor='black', alpha=0.7)
plt.axvline(np.mean(eval_rewards), color='red', linestyle='--', label=f'Mean = {np.mean(eval_rewards):.1f}')
plt.title('Evaluation Reward Distribution')
plt.xlabel('Episode Reward')
plt.ylabel('Count')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## 11. Save & Load Weights

Weights are auto-saved during training whenever a new best success rate is achieved.  
You can also save/load manually below.

In [ ]:
# Manual save
dqn_agent.save_weights_file_path = 'weights/model.h5'
dqn_agent.save_weights()

In [ ]:
# Manual load (into the current agent)
dqn_agent.load_weights_file_path = 'weights/model.h5'
dqn_agent._load_weights()
print('Weights loaded successfully.')

---
## 12. Watch a Trained Episode

After training, run a single episode to see the full conversation.

In [ ]:
dqn_agent.eps = 0.0  # No exploration

state_tracker.reset()
user_action = user_sim.reset()
goal = copy.deepcopy(user_sim.goal)
emc.infuse_error(user_action)
state_tracker.update_state_user(user_action)
dqn_agent.reset()

print('=' * 60)
print('USER GOAL')
print(f"  Inform : {goal['inform_slots']}")
print(f"  Request: {goal['request_slots']}")
print('-' * 60)
print(f'[User ] intent={user_action["intent"]}  inform={user_action["inform_slots"]}  request={user_action["request_slots"]}')

done = False
total_reward = 0
while not done:
    state = state_tracker.get_state()
    action_index, agent_action = dqn_agent.get_action(state)
    state_tracker.update_state_agent(agent_action)
    print(f'[Agent] intent={agent_action["intent"]}  inform={agent_action["inform_slots"]}  request={agent_action["request_slots"]}')
    
    user_action, reward, done, success = user_sim.step(agent_action)
    total_reward += reward
    if not done:
        emc.infuse_error(user_action)
    state_tracker.update_state_user(user_action)
    print(f'[User ] intent={user_action["intent"]}  inform={user_action["inform_slots"]}  request={user_action["request_slots"]}')

print('-' * 60)
print(f'Result: {"SUCCESS" if success else "FAILURE"}  |  Total reward: {total_reward}')
print('=' * 60)

---
## 13. Experiment Log

Use this cell to keep notes on experiments you run.

In [ ]:
experiment_log = {
    'description': 'Baseline DQN — single hidden layer 80 units',
    'num_episodes': NUM_EP_TRAIN,
    'best_success_rate': success_rate_best,
    'final_epsilon': dqn_agent.eps,
    'eval_success_rate': eval_successes / NUM_EVAL if 'eval_successes' in dir() else 'N/A',
    'config': config,
}

# Save experiment log
with open('experiment_log.json', 'w') as f:
    json.dump(experiment_log, f, indent=2, default=str)

print('Experiment log saved to experiment_log.json')
print(json.dumps(experiment_log, indent=2, default=str))